In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report



df = pd.read_csv('/content/data_for_regression.csv')

X_text = df["text"]
y = df["label"]

print("Dataset shape:", df.shape)
print("Label distribution:\n", y.value_counts())


X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)


tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.9,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_val_tfidf = tfidf_vectorizer.transform(X_val_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)



lr_model = LogisticRegression(
    max_iter=1000,
    solver="liblinear"
)

lr_param_dist = {
    "C": np.logspace(-3, 2, 20),
    "penalty": ["l1", "l2"],
    "class_weight": [None, "balanced"]
}

lr_search = RandomizedSearchCV(
    estimator=lr_model,
    param_distributions=lr_param_dist,
    n_iter=20,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

print("\nTuning Logistic Regression...")
lr_search.fit(X_train_tfidf, y_train)

best_lr = lr_search.best_estimator_

print("\nBest Logistic Regression parameters:")
print(lr_search.best_params_)

print("\nLogistic Regression — Validation Results")
print(classification_report(y_val, best_lr.predict(X_val_tfidf)))



svm_model = LinearSVC(
    dual=False,
    max_iter=5000
)

svm_param_dist = {
    "C": np.logspace(-3, 2, 20),
    "class_weight": [None, "balanced"]
}

svm_search = RandomizedSearchCV(
    estimator=svm_model,
    param_distributions=svm_param_dist,
    n_iter=20,
    scoring="f1",
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=2
)

print("\nTuning Linear SVM...")
svm_search.fit(X_train_tfidf, y_train)

best_svm = svm_search.best_estimator_

print("\nBest Linear SVM parameters:")
print(svm_search.best_params_)

print("\nLinear SVM — Validation Results")
print(classification_report(y_val, best_svm.predict(X_val_tfidf)))



print("\nFINAL TEST RESULTS — Logistic Regression")
print(classification_report(y_test, best_lr.predict(X_test_tfidf)))

print("\nFINAL TEST RESULTS — Linear SVM")
print(classification_report(y_test, best_svm.predict(X_test_tfidf)))


Dataset shape: (40431, 4)
Label distribution:
 label
0    20216
1    20215
Name: count, dtype: int64

Tuning Logistic Regression...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best Logistic Regression parameters:
{'penalty': 'l2', 'class_weight': 'balanced', 'C': np.float64(2.636650898730358)}

Logistic Regression — Validation Results
              precision    recall  f1-score   support

           0       0.90      0.90      0.90      2022
           1       0.90      0.90      0.90      2021

    accuracy                           0.90      4043
   macro avg       0.90      0.90      0.90      4043
weighted avg       0.90      0.90      0.90      4043


Tuning Linear SVM...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best Linear SVM parameters:
{'class_weight': 'balanced', 'C': np.float64(0.23357214690901212)}

Linear SVM — Validation Results
              precision    recall  f1-score   support

           0       0.90      0.90      0.90      